In [1]:
import xarray as xr
import rioxarray as rxr
import numpy as np
import geopandas as gpd

In [2]:
from disturbance_integration import (
    majority_filter,
    minimum_mapping_unit_filter,
    add_zonal_statistic,
    count_points_within_zones,
    compute_disturbance_index,
)

In [ ]:
# Disturbance Index inputs
ZONES_POLYS = "/home/hm/downloads/gnp_grid_small.geojson"
BREAKS_RASTER = "/home/hm/downloads/garamba_savi_bap_breaks_cloud_le70_smallclip.tif"
FIRES_POINTS = "/home/hm/downloads/fires_smallclip.geojson"
BUILT_RASTER = "/home/hm/downloads/garamba_built_smallclip.tif"

In [ ]:
# Load zones polygons
zones = gpd.read_file(ZONES_POLYS)  # get aggregation zones

In [ ]:
# read breaks raster
breaks_raster = rxr.open_rasterio(BREAKS_RASTER)  # get breaks raster
breaks_raster = breaks_raster.where(breaks_raster != breaks_raster.rio.nodata)
years, magnitude = breaks_raster

In [ ]:
# filter on magnitude but keep nodata (no detection)
years = ((magnitude <= -200) | (magnitude == magnitude.rio.nodata)) * years

In [ ]:
# majority with opencv requires int types
years = years.fillna(1).astype(np.uint16)

In [ ]:
%%time
# breaks mode convolution
filtered_years = majority_filter(years, filter_size=7, use_opencv=True)
filtered_years = filtered_years.rio.write_crs(years.rio.crs)  # mmu requires projection

In [ ]:
%%time
# breaks MMU - needs to be an integer dtype
mmu_filtered = minimum_mapping_unit_filter(filtered_years.astype(np.int16), mmu_area=5000, connectivity=8)  # MMU sieve filter
# just to avoid warning about nodata not set. 0 is a valid value
mmu_filtered = mmu_filtered.rio.write_nodata(1)

In [ ]:
# float so we can null no detections
mmu_filtered = mmu_filtered.where(mmu_filtered > 0)

In [ ]:
# Load other components
fires = gpd.read_file(FIRES_POINTS)  # get FIRMS points
built_s = rxr.open_rasterio(BUILT_RASTER)[0]  # get built-up raster

In [ ]:
# zonal stats
zones = add_zonal_statistic(zones=zones, raster=mmu_filtered, statistic='count', output_field_label='disturbance')  # add disturbance summary
zones = count_points_within_zones(zones=zones, points=fires, count_column='fire_count')  # add fire summary
zones = add_zonal_statistic(zones=zones, raster=built_s, statistic='sum', output_field_label='built')  # add built-up summary

In [ ]:
# Compute disturbance index
zones['disturbance_index'] = compute_disturbance_index(zones.iloc[:,-3:])  # send last three columns we've appended for normalization and index computation
zones